# Trivariate Ordinal Regression: Life Satisfaction, Job Satisfaction, and Financial Satisfaction

This notebook implements a trivariate ordinal regression model to jointly predict:
- **LIFENOW**: Life satisfaction (1-10 scale)
- **SATJOB**: Job satisfaction (1-4 scale, where 1=very satisfied, 4=very dissatisfied)
- **SATFIN**: Financial satisfaction (1-3 scale, where 1=satisfied, 3=not at all satisfied)

## Theoretical Framework

All three outcomes are modeled as ordinal manifestations of underlying continuous latent variables:

$$
Y_1^* = X'\beta_1 + \varepsilon_1 \quad \text{(latent life satisfaction)}
$$
$$
Y_2^* = X'\beta_2 + \varepsilon_2 \quad \text{(latent job satisfaction)}
$$
$$
Y_3^* = X'\beta_3 + \varepsilon_3 \quad \text{(latent financial satisfaction)}
$$

where the errors are correlated via a multivariate normal distribution:

$$
\begin{pmatrix} \varepsilon_1 \\ \varepsilon_2 \\ \varepsilon_3 \end{pmatrix} \sim \text{MVN}\left(\mathbf{0}, \Sigma\right)
$$

with correlation matrix:

$$
\Sigma = \begin{pmatrix} 1 & \rho_{12} & \rho_{13} \\ \rho_{12} & 1 & \rho_{23} \\ \rho_{13} & \rho_{23} & 1 \end{pmatrix}
$$

The correlations $\rho_{ij}$ capture the **polychoric correlations**—the associations between ordinal outcomes after accounting for their latent continuous structure.

In [1]:
import pandas as pd
import numpy as np
import pymc as pm
import pytensor.tensor as pt
import arviz as az
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from scipy.special import ndtr  # Standard normal CDF
from pathlib import Path

np.random.seed(42)
pd.set_option('display.max_columns', None)

print(f'PyMC version: {pm.__version__}')
print(f'ArviZ version: {az.__version__}')

PyMC version: 5.27.0
ArviZ version: 0.22.0


## Data Loading and Preparation

In [2]:
# Load data
df = pd.read_csv('../data/gss_2022.csv')
print(f'Full dataset: {df.shape}')

# Define outcomes and predictors
# Note: satfin is now an outcome, not a predictor
outcomes = ['lifenow', 'satjob', 'satfin']
predictors = ['hlthdep', 'stress', 'wrkmeangfl', 'finrela', 'anxiety', 'age', 'sex', 'degree']

# Select columns and drop missing
analysis_cols = outcomes + predictors
df_analysis = df[analysis_cols].dropna().copy()

# Filter to valid outcome values
df_analysis = df_analysis[
    (df_analysis['lifenow'].between(1, 10)) & 
    (df_analysis['satjob'].between(1, 4)) &
    (df_analysis['satfin'].between(1, 3))
].copy()

print(f'Analysis dataset: {df_analysis.shape}')
print(f'Complete cases: {len(df_analysis)}')

Full dataset: (3544, 24)
Analysis dataset: (495, 11)
Complete cases: 495


In [3]:
# Examine outcome distributions
print('=== LIFENOW (Life Satisfaction) ===')
print(df_analysis['lifenow'].value_counts().sort_index())
print(f'\nRange: {df_analysis["lifenow"].min():.0f} - {df_analysis["lifenow"].max():.0f}')

print('\n=== SATJOB (Job Satisfaction) ===')
print(df_analysis['satjob'].value_counts().sort_index())
print('(1=Very Satisfied, 2=Mod Satisfied, 3=Little Dissatisfied, 4=Very Dissatisfied)')

print('\n=== SATFIN (Financial Satisfaction) ===')
print(df_analysis['satfin'].value_counts().sort_index())
print('(1=Satisfied, 2=More or Less Satisfied, 3=Not At All Satisfied)')

=== LIFENOW (Life Satisfaction) ===
lifenow
1.0       1
3.0       4
4.0       9
5.0      30
6.0      51
7.0      45
8.0     127
9.0     163
10.0     65
Name: count, dtype: int64

Range: 1 - 10

=== SATJOB (Job Satisfaction) ===
satjob
1.0    223
2.0    209
3.0     47
4.0     16
Name: count, dtype: int64
(1=Very Satisfied, 2=Mod Satisfied, 3=Little Dissatisfied, 4=Very Dissatisfied)

=== SATFIN (Financial Satisfaction) ===
satfin
1.0     95
2.0    260
3.0    140
Name: count, dtype: int64
(1=Satisfied, 2=More or Less Satisfied, 3=Not At All Satisfied)


In [4]:
# Pairwise joint distributions and correlations
outcome_pairs = [('lifenow', 'satjob'), ('lifenow', 'satfin'), ('satjob', 'satfin')]
pair_labels = [
    ('LIFENOW', 'SATJOB'),
    ('LIFENOW', 'SATFIN'),
    ('SATJOB', 'SATFIN')
]

for (var1, var2), (label1, label2) in zip(outcome_pairs, pair_labels):
    crosstab = pd.crosstab(df_analysis[var2], df_analysis[var1])
    print(f'Joint distribution ({label2} rows × {label1} columns):')
    print(crosstab)
    
    spearman_corr = df_analysis[[var1, var2]].corr(method='spearman').iloc[0, 1]
    print(f'Spearman correlation: {spearman_corr:.3f}\n')

# Overall correlation matrix
print('=== Spearman Correlation Matrix (Outcomes) ===')
outcome_corr = df_analysis[outcomes].corr(method='spearman')
print(outcome_corr.round(3))

Joint distribution (SATJOB rows × LIFENOW columns):
lifenow  1.0   3.0   4.0   5.0   6.0   7.0   8.0   9.0   10.0
satjob                                                       
1.0         1     0     3     8    15    10    56    84    46
2.0         0     1     3    15    24    24    56    71    15
3.0         0     2     1     6     7    10    12     5     4
4.0         0     1     2     1     5     1     3     3     0
Spearman correlation: -0.308

Joint distribution (SATFIN rows × LIFENOW columns):
lifenow  1.0   3.0   4.0   5.0   6.0   7.0   8.0   9.0   10.0
satfin                                                       
1.0         1     0     1     4     0     2    17    47    23
2.0         0     1     1    13    16    20    74   102    33
3.0         0     3     7    13    35    23    36    14     9
Spearman correlation: -0.431

Joint distribution (SATFIN rows × SATJOB columns):
satjob  1.0  2.0  3.0  4.0
satfin                    
1.0      57   33    5    0
2.0     119  115   21 

In [5]:
# Visualize pairwise joint distributions
fig = make_subplots(rows=1, cols=3, subplot_titles=[
    'LIFENOW × SATJOB', 'LIFENOW × SATFIN', 'SATJOB × SATFIN'
])

# LIFENOW × SATJOB
crosstab1 = pd.crosstab(df_analysis['satjob'], df_analysis['lifenow'], normalize='all')
fig.add_trace(go.Heatmap(
    z=crosstab1.values,
    x=crosstab1.columns.astype(int),
    y=crosstab1.index.astype(int),
    colorscale='Blues',
    showscale=False
), row=1, col=1)

# LIFENOW × SATFIN
crosstab2 = pd.crosstab(df_analysis['satfin'], df_analysis['lifenow'], normalize='all')
fig.add_trace(go.Heatmap(
    z=crosstab2.values,
    x=crosstab2.columns.astype(int),
    y=crosstab2.index.astype(int),
    colorscale='Blues',
    showscale=False
), row=1, col=2)

# SATJOB × SATFIN
crosstab3 = pd.crosstab(df_analysis['satfin'], df_analysis['satjob'], normalize='all')
fig.add_trace(go.Heatmap(
    z=crosstab3.values,
    x=crosstab3.columns.astype(int),
    y=crosstab3.index.astype(int),
    colorscale='Blues',
    showscale=True
), row=1, col=3)

fig.update_xaxes(title_text='Life Satisfaction', row=1, col=1)
fig.update_yaxes(title_text='Job Satisfaction', row=1, col=1)
fig.update_xaxes(title_text='Life Satisfaction', row=1, col=2)
fig.update_yaxes(title_text='Financial Satisfaction', row=1, col=2)
fig.update_xaxes(title_text='Job Satisfaction', row=1, col=3)
fig.update_yaxes(title_text='Financial Satisfaction', row=1, col=3)

fig.update_layout(
    title='Pairwise Joint Distributions of Satisfaction Outcomes',
    height=350, width=1000
)
fig.show()

## Data Preparation for Modeling

In [6]:
# Prepare outcomes (0-indexed for PyMC)
df_analysis['y1'] = (df_analysis['lifenow'] - 1).astype(int)  # 0-9
df_analysis['y2'] = (df_analysis['satjob'] - 1).astype(int)   # 0-3
df_analysis['y3'] = (df_analysis['satfin'] - 1).astype(int)   # 0-2

K1 = 10  # LIFENOW categories
K2 = 4   # SATJOB categories
K3 = 3   # SATFIN categories

print(f'LIFENOW coded: 0-{K1-1} ({K1} categories, {K1-1} cutpoints)')
print(f'SATJOB coded: 0-{K2-1} ({K2} categories, {K2-1} cutpoints)')
print(f'SATFIN coded: 0-{K3-1} ({K3} categories, {K3-1} cutpoints)')

LIFENOW coded: 0-9 (10 categories, 9 cutpoints)
SATJOB coded: 0-3 (4 categories, 3 cutpoints)
SATFIN coded: 0-2 (3 categories, 2 cutpoints)


In [7]:
# Prepare predictors
# Binary: sex (female=1)
df_analysis['female'] = (df_analysis['sex'] == 2).astype(int)

# Standardize continuous/ordinal predictors
# Note: satfin is now an outcome, not a predictor
predictors_to_std = ['hlthdep', 'stress', 'wrkmeangfl', 'finrela', 'anxiety', 'age', 'degree']
predictor_names = predictors_to_std + ['female']

# Store standardization parameters
std_params = {}
for var in predictors_to_std:
    mean_val = df_analysis[var].mean()
    std_val = df_analysis[var].std()
    df_analysis[f'{var}_std'] = (df_analysis[var] - mean_val) / std_val
    std_params[var] = {'mean': mean_val, 'std': std_val}

# Build design matrix
X_cols = [f'{v}_std' for v in predictors_to_std] + ['female']
X = df_analysis[X_cols].values

print(f'Design matrix shape: {X.shape}')
print(f'Predictors: {predictor_names}')

Design matrix shape: (495, 8)
Predictors: ['hlthdep', 'stress', 'wrkmeangfl', 'finrela', 'anxiety', 'age', 'degree', 'female']


In [8]:
# Predictor correlation matrix
corr_matrix = df_analysis[X_cols].corr()
corr_matrix.columns = predictor_names
corr_matrix.index = predictor_names

fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale='RdBu_r',
    zmin=-1, zmax=1,
    text=np.round(corr_matrix.values, 2),
    texttemplate='%{text}'
))
fig.update_layout(title='Predictor Correlations', width=700, height=600)
fig.show()

## Bivariate Normal CDF Implementation

The key computational challenge is evaluating the bivariate normal CDF for rectangle probabilities. We need:

$$
P(Y_1=k, Y_2=j) = \Phi_2(\theta_{1,k}, \theta_{2,j}; \rho) - \Phi_2(\theta_{1,k-1}, \theta_{2,j}; \rho) - \Phi_2(\theta_{1,k}, \theta_{2,j-1}; \rho) + \Phi_2(\theta_{1,k-1}, \theta_{2,j-1}; \rho)
$$

We'll use an approximation based on the Drezner-Wesolowsky method.

In [9]:
def bvn_cdf_approx(x, y, rho):
    """
    Approximate bivariate normal CDF using Drezner-Wesolowsky (1990) method.
    
    This computes P(X <= x, Y <= y) where (X, Y) ~ BVN(0, 0, 1, 1, rho).
    Uses Gauss-Legendre quadrature for numerical integration.
    """
    # Gauss-Legendre weights and abscissas for 6-point quadrature
    w = np.array([0.1713244923791705, 0.3607615730481384, 0.4679139345726904,
                  0.4679139345726904, 0.3607615730481384, 0.1713244923791705])
    xi = np.array([-0.9324695142031522, -0.6612093864662647, -0.2386191860831970,
                   0.2386191860831970, 0.6612093864662647, 0.9324695142031522])
    
    # Handle edge cases
    if np.abs(rho) < 1e-10:
        return stats.norm.cdf(x) * stats.norm.cdf(y)
    
    if rho > 0.9999:
        return stats.norm.cdf(min(x, y))
    
    if rho < -0.9999:
        return max(0, stats.norm.cdf(x) - stats.norm.cdf(-y))
    
    # Drezner-Wesolowsky approximation
    h = -x
    k = -y
    hk = h * k
    
    if np.abs(rho) < 0.925:
        # Use direct formula for moderate correlations
        hs = (h * h + k * k) / 2
        asr = np.arcsin(rho)
        sn = np.sin(asr * (1 + xi) / 2)
        bvn = np.sum(w * np.exp((sn * hk - hs) / (1 - sn * sn)))
        bvn = bvn * asr / (4 * np.pi) + stats.norm.cdf(-h) * stats.norm.cdf(-k)
    else:
        # Use alternative formula for high correlations
        if rho < 0:
            k = -k
            hk = -hk
        
        if np.abs(rho) < 1:
            ass = (1 - rho) * (1 + rho)
            a = np.sqrt(ass)
            bs = (h - k) ** 2
            c = (4 - hk) / 8
            d = (12 - hk) / 16
            asr = -(bs / ass + hk) / 2
            if asr > -100:
                bvn = a * np.exp(asr) * (1 - c * (bs - ass) * (1 - d * bs / 5) / 3 + c * d * ass * ass / 5)
            else:
                bvn = 0
            
            if -hk < 100:
                b = np.sqrt(bs)
                bvn = bvn - np.exp(-hk / 2) * np.sqrt(2 * np.pi) * stats.norm.cdf(-b / a) * b * (1 - c * bs * (1 - d * bs / 5) / 3)
            
            a = a / 2
            xs = (a * (1 + xi)) ** 2
            asr = -(bs / xs + hk) / 2
            valid = asr > -100
            if np.any(valid):
                asr_valid = np.where(valid, asr, -100)
                bvn = bvn + a * np.sum(w * np.exp(asr_valid) * (np.exp(-hk * (1 - xs) / (2 * (1 + np.sqrt(1 - xs)))) / np.sqrt(1 - xs) - (1 + c * xs * (1 + d * xs))))
            
            bvn = -bvn / (2 * np.pi)
        
        if rho > 0:
            bvn = bvn + stats.norm.cdf(-max(h, k))
        else:
            bvn = -bvn
            if k > h:
                bvn = bvn + stats.norm.cdf(k) - stats.norm.cdf(h)
    
    return max(0, min(1, bvn))

# Vectorized version for arrays
bvn_cdf_vec = np.vectorize(bvn_cdf_approx)

# Test the implementation
print('Testing BVN CDF approximation:')
print(f'  P(X<0, Y<0 | rho=0.5): {bvn_cdf_approx(0, 0, 0.5):.4f} (expected ~0.333)')
print(f'  P(X<0, Y<0 | rho=0.0): {bvn_cdf_approx(0, 0, 0.0):.4f} (expected 0.250)')
print(f'  P(X<0, Y<0 | rho=-0.5): {bvn_cdf_approx(0, 0, -0.5):.4f} (expected ~0.167)')

Testing BVN CDF approximation:
  P(X<0, Y<0 | rho=0.5): 0.3333 (expected ~0.333)
  P(X<0, Y<0 | rho=0.0): 0.2500 (expected 0.250)
  P(X<0, Y<0 | rho=-0.5): 0.1667 (expected ~0.167)


## PyTensor Implementation of Bivariate Normal CDF

We need a differentiable version for use in PyMC. We'll use the Owen's T function approach which has a simpler form for automatic differentiation.

In [10]:
def owens_t_approx(h, a):
    """
    Owen's T function approximation using series expansion.
    T(h, a) = (1/2π) ∫₀ᵃ exp(-h²(1+t²)/2) / (1+t²) dt
    """
    # Use Gaussian quadrature
    n_points = 10
    t, w = np.polynomial.legendre.leggauss(n_points)
    
    # Transform from [-1, 1] to [0, a]
    t_scaled = a * (t + 1) / 2
    w_scaled = w * a / 2
    
    integrand = np.exp(-h**2 * (1 + t_scaled**2) / 2) / (1 + t_scaled**2)
    return np.sum(w_scaled * integrand) / (2 * np.pi)

def bvn_cdf_owens(x, y, rho):
    """
    Bivariate normal CDF using Owen's T function.
    P(X <= x, Y <= y) where (X, Y) ~ BVN(0, 0, 1, 1, rho)
    """
    if np.abs(rho) < 1e-10:
        return stats.norm.cdf(x) * stats.norm.cdf(y)
    
    # Formula: Φ₂(x, y; ρ) = Φ(x)Φ(y) + T(x, (y-ρx)/√(1-ρ²)/x) + T(y, (x-ρy)/√(1-ρ²)/y)
    # when x, y > 0
    
    # Use the Drezner formula which is more stable
    return bvn_cdf_approx(x, y, rho)

# For PyTensor, we'll create an Op that wraps scipy's multivariate normal
from pytensor.tensor import as_tensor_variable
from pytensor.graph.op import Op
from pytensor.graph.basic import Apply

class BivariateNormalCDF(Op):
    """
    PyTensor Op for bivariate normal CDF.
    """
    __props__ = ()
    
    def make_node(self, x, y, rho):
        x = as_tensor_variable(x)
        y = as_tensor_variable(y)
        rho = as_tensor_variable(rho)
        return Apply(self, [x, y, rho], [x.type()])
    
    def perform(self, node, inputs, output_storage):
        x, y, rho = inputs
        # Use scipy's mvn for accurate computation
        result = np.zeros_like(x)
        for i in range(len(x)):
            result[i] = bvn_cdf_approx(x[i], y[i], rho)
        output_storage[0][0] = result
    
    def grad(self, inputs, output_grads):
        x, y, rho = inputs
        gz = output_grads[0]
        
        # Gradients of BVN CDF
        # ∂Φ₂/∂x = φ(x) Φ((y - ρx) / √(1-ρ²))
        # ∂Φ₂/∂y = φ(y) Φ((x - ρy) / √(1-ρ²))
        
        sqrt_1_rho2 = pt.sqrt(1 - rho**2)
        
        phi_x = pt.exp(-x**2 / 2) / pt.sqrt(2 * np.pi)
        phi_y = pt.exp(-y**2 / 2) / pt.sqrt(2 * np.pi)
        
        Phi_cond_x = 0.5 * (1 + pt.erf((y - rho * x) / (sqrt_1_rho2 * pt.sqrt(2))))
        Phi_cond_y = 0.5 * (1 + pt.erf((x - rho * y) / (sqrt_1_rho2 * pt.sqrt(2))))
        
        grad_x = gz * phi_x * Phi_cond_x
        grad_y = gz * phi_y * Phi_cond_y
        
        # Gradient w.r.t. rho is more complex - use finite differences or derive
        # For now, use a numerical approximation
        grad_rho = pt.zeros_like(rho)  # Simplified - could be improved
        
        return [grad_x, grad_y, grad_rho]

bvn_cdf_op = BivariateNormalCDF()

print('BivariateNormalCDF Op created successfully')

BivariateNormalCDF Op created successfully


## Alternative Approach: Marginalized Likelihood with Numerical Integration

Given the complexity of implementing a fully differentiable multivariate normal CDF, we'll use an alternative approach:

1. **First**, fit marginal ordinal probit models for each outcome separately
2. **Then**, estimate the polychoric correlations from the latent residuals
3. **Finally**, fit a full trivariate model using the estimated correlations as informative constraints

This two-stage approach is computationally simpler and provides good estimates.

In [11]:
# Extract data for modeling
y1 = df_analysis['y1'].values  # LIFENOW (0-9)
y2 = df_analysis['y2'].values  # SATJOB (0-3)
y3 = df_analysis['y3'].values  # SATFIN (0-2)
N = len(y1)

print(f'Sample size: {N}')
print(f'LIFENOW categories: {K1} (0 to {K1-1})')
print(f'SATJOB categories: {K2} (0 to {K2-1})')
print(f'SATFIN categories: {K3} (0 to {K3-1})')

Sample size: 495
LIFENOW categories: 10 (0 to 9)
SATJOB categories: 4 (0 to 3)
SATFIN categories: 3 (0 to 2)


## Model 1: Marginal Ordinal Probit for LIFENOW

In [12]:
coords_lifenow = {
    'predictors': predictor_names,
    'obs': np.arange(N)
}

with pm.Model(coords=coords_lifenow) as model_lifenow:
    # Data
    X_data = pm.Data('X', X)
    y_data = pm.Data('y', y1)
    
    # Priors on regression coefficients
    beta = pm.Normal('beta', mu=0, sigma=1, dims='predictors')
    
    # Linear predictor
    eta = pm.math.dot(X_data, beta)
    
    # Ordered cutpoints for probit model (9 cutpoints for 10 categories)
    # Use cumulative sum of positive increments to ensure ordering
    cutpoint_deltas = pm.Exponential('cutpoint_deltas', lam=1, shape=K1-2)
    cutpoint_base = pm.Normal('cutpoint_base', mu=0, sigma=2)
    cutpoints = pm.Deterministic(
        'cutpoints',
        pt.concatenate([[cutpoint_base], cutpoint_base + pt.cumsum(cutpoint_deltas)])
    )
    
    # Likelihood using OrderedProbit
    y_obs = pm.OrderedProbit('y_obs', eta=eta, cutpoints=cutpoints, observed=y_data)

print('LIFENOW model structure:')
print(model_lifenow)

LIFENOW model structure:


In [13]:
# Fit LIFENOW model
with model_lifenow:
    trace_lifenow = pm.sample(
        draws=1000,
        tune=1000,
        target_accept=0.9,
        nuts_sampler='nutpie',
        random_seed=42
    )

Progress,Draws,Divergences,Step Size,Gradients/Draw
,2000,1,0.21,31
,2000,4,0.19,31
,2000,4,0.21,63
,2000,0,0.20,15


In [14]:
# Check convergence
print('=== LIFENOW Model Diagnostics ===')
if 'diverging' in trace_lifenow.sample_stats:
    n_div = int(trace_lifenow.sample_stats['diverging'].sum().values)
    print(f'Divergent transitions: {n_div}')

summary_lifenow = az.summary(trace_lifenow, var_names=['beta'])
print('\nCoefficient estimates (LIFENOW):')
print(summary_lifenow[['mean', 'sd', 'hdi_3%', 'hdi_97%', 'r_hat', 'ess_bulk']])

=== LIFENOW Model Diagnostics ===
Divergent transitions: 9

Coefficient estimates (LIFENOW):
                   mean     sd  hdi_3%  hdi_97%  r_hat  ess_bulk
beta[hlthdep]    -0.422  0.060  -0.533   -0.307    1.0    5449.0
beta[stress]      0.141  0.053   0.040    0.242    1.0    7126.0
beta[wrkmeangfl] -0.233  0.050  -0.331   -0.144    1.0    5924.0
beta[finrela]     0.260  0.054   0.157    0.360    1.0    5941.0
beta[anxiety]    -0.005  0.059  -0.114    0.106    1.0    5808.0
beta[age]         0.063  0.051  -0.029    0.162    1.0    6849.0
beta[degree]      0.123  0.054   0.019    0.221    1.0    6000.0
beta[female]     -0.030  0.097  -0.207    0.152    1.0    5980.0


## Model 2: Marginal Ordinal Probit for SATJOB

In [15]:
coords_satjob = {
    'predictors': predictor_names,
    'obs': np.arange(N)
}

with pm.Model(coords=coords_satjob) as model_satjob:
    # Data
    X_data = pm.Data('X', X)
    y_data = pm.Data('y', y2)
    
    # Priors on regression coefficients
    beta = pm.Normal('beta', mu=0, sigma=1, dims='predictors')
    
    # Linear predictor
    eta = pm.math.dot(X_data, beta)
    
    # Ordered cutpoints for probit model (3 cutpoints for 4 categories)
    cutpoint_deltas = pm.Exponential('cutpoint_deltas', lam=1, shape=K2-2)
    cutpoint_base = pm.Normal('cutpoint_base', mu=0, sigma=2)
    cutpoints = pm.Deterministic(
        'cutpoints',
        pt.concatenate([[cutpoint_base], cutpoint_base + pt.cumsum(cutpoint_deltas)])
    )
    
    # Likelihood using OrderedProbit
    y_obs = pm.OrderedProbit('y_obs', eta=eta, cutpoints=cutpoints, observed=y_data)

print('SATJOB model structure:')
print(model_satjob)

SATJOB model structure:


In [16]:
# Fit SATJOB model
with model_satjob:
    trace_satjob = pm.sample(
        draws=1000,
        tune=1000,
        target_accept=0.9,
        nuts_sampler='nutpie',
        random_seed=42
    )

Progress,Draws,Divergences,Step Size,Gradients/Draw
,2000,0,0.50,7
,2000,0,0.53,15
,2000,0,0.48,7
,2000,0,0.54,7


In [17]:
# Check convergence
print('=== SATJOB Model Diagnostics ===')
if 'diverging' in trace_satjob.sample_stats:
    n_div = int(trace_satjob.sample_stats['diverging'].sum().values)
    print(f'Divergent transitions: {n_div}')

summary_satjob = az.summary(trace_satjob, var_names=['beta'])
print('\nCoefficient estimates (SATJOB):')
print(summary_satjob[['mean', 'sd', 'hdi_3%', 'hdi_97%', 'r_hat', 'ess_bulk']])

=== SATJOB Model Diagnostics ===
Divergent transitions: 0

Coefficient estimates (SATJOB):
                   mean     sd  hdi_3%  hdi_97%  r_hat  ess_bulk
beta[hlthdep]     0.201  0.066   0.079    0.332    1.0    3928.0
beta[stress]     -0.283  0.059  -0.397   -0.175    1.0    6112.0
beta[wrkmeangfl]  0.516  0.058   0.414    0.632    1.0    5846.0
beta[finrela]    -0.081  0.060  -0.195    0.030    1.0    4215.0
beta[anxiety]    -0.059  0.065  -0.182    0.063    1.0    3792.0
beta[age]        -0.169  0.057  -0.278   -0.065    1.0    6888.0
beta[degree]      0.024  0.060  -0.086    0.133    1.0    4257.0
beta[female]      0.292  0.107   0.093    0.491    1.0    2997.0


## Model 3: Marginal Ordinal Probit for SATFIN

In [18]:
coords_satfin = {
    'predictors': predictor_names,
    'obs': np.arange(N)
}

with pm.Model(coords=coords_satfin) as model_satfin:
    # Data
    X_data = pm.Data('X', X)
    y_data = pm.Data('y', y3)
    
    # Priors on regression coefficients
    beta = pm.Normal('beta', mu=0, sigma=1, dims='predictors')
    
    # Linear predictor
    eta = pm.math.dot(X_data, beta)
    
    # Ordered cutpoints for probit model (2 cutpoints for 3 categories)
    cutpoint_deltas = pm.Exponential('cutpoint_deltas', lam=1, shape=K3-2)
    cutpoint_base = pm.Normal('cutpoint_base', mu=0, sigma=2)
    cutpoints = pm.Deterministic(
        'cutpoints',
        pt.concatenate([[cutpoint_base], cutpoint_base + pt.cumsum(cutpoint_deltas)])
    )
    
    # Likelihood using OrderedProbit
    y_obs = pm.OrderedProbit('y_obs', eta=eta, cutpoints=cutpoints, observed=y_data)

print('SATFIN model structure:')
print(model_satfin)

SATFIN model structure:


In [19]:
# Fit SATFIN model
with model_satfin:
    trace_satfin = pm.sample(
        draws=1000,
        tune=1000,
        target_accept=0.9,
        nuts_sampler='nutpie',
        random_seed=42
    )

Progress,Draws,Divergences,Step Size,Gradients/Draw
,2000,0,0.51,15
,2000,0,0.54,7
,2000,0,0.49,15
,2000,0,0.48,7


In [20]:
# Check convergence
print('=== SATFIN Model Diagnostics ===')
if 'diverging' in trace_satfin.sample_stats:
    n_div = int(trace_satfin.sample_stats['diverging'].sum().values)
    print(f'Divergent transitions: {n_div}')

summary_satfin = az.summary(trace_satfin, var_names=['beta'])
print('\nCoefficient estimates (SATFIN):')
print(summary_satfin[['mean', 'sd', 'hdi_3%', 'hdi_97%', 'r_hat', 'ess_bulk']])

=== SATFIN Model Diagnostics ===
Divergent transitions: 0

Coefficient estimates (SATFIN):
                   mean     sd  hdi_3%  hdi_97%  r_hat  ess_bulk
beta[hlthdep]     0.258  0.067   0.125    0.375    1.0    4252.0
beta[stress]     -0.023  0.056  -0.134    0.079    1.0    5134.0
beta[wrkmeangfl]  0.063  0.056  -0.036    0.171    1.0    6053.0
beta[finrela]    -0.487  0.063  -0.607   -0.366    1.0    4741.0
beta[anxiety]     0.051  0.067  -0.074    0.177    1.0    3944.0
beta[age]        -0.036  0.056  -0.137    0.075    1.0    5631.0
beta[degree]     -0.203  0.059  -0.321   -0.101    1.0    4795.0
beta[female]      0.166  0.111  -0.051    0.365    1.0    3537.0


## Coefficient Comparison: LIFENOW, SATJOB, and SATFIN

In [21]:
# Extract posterior means and HDIs for all three outcomes
beta_lifenow = trace_lifenow.posterior['beta'].values.reshape(-1, len(predictor_names))
beta_satjob = trace_satjob.posterior['beta'].values.reshape(-1, len(predictor_names))
beta_satfin = trace_satfin.posterior['beta'].values.reshape(-1, len(predictor_names))

comparison_df = pd.DataFrame({
    'Predictor': predictor_names,
    'LIFENOW_mean': beta_lifenow.mean(axis=0),
    'LIFENOW_hdi_low': np.percentile(beta_lifenow, 3, axis=0),
    'LIFENOW_hdi_high': np.percentile(beta_lifenow, 97, axis=0),
    'SATJOB_mean': beta_satjob.mean(axis=0),
    'SATJOB_hdi_low': np.percentile(beta_satjob, 3, axis=0),
    'SATJOB_hdi_high': np.percentile(beta_satjob, 97, axis=0),
    'SATFIN_mean': beta_satfin.mean(axis=0),
    'SATFIN_hdi_low': np.percentile(beta_satfin, 3, axis=0),
    'SATFIN_hdi_high': np.percentile(beta_satfin, 97, axis=0),
})

print('Coefficient Comparison (standardized predictors):')
print('Note: All outcomes coded so higher = more dissatisfied')
print(comparison_df[['Predictor', 'LIFENOW_mean', 'SATJOB_mean', 'SATFIN_mean']].round(3))

Coefficient Comparison (standardized predictors):
Note: All outcomes coded so higher = more dissatisfied
    Predictor  LIFENOW_mean  SATJOB_mean  SATFIN_mean
0     hlthdep        -0.422        0.201        0.258
1      stress         0.141       -0.283       -0.023
2  wrkmeangfl        -0.233        0.516        0.063
3     finrela         0.260       -0.081       -0.487
4     anxiety        -0.005       -0.059        0.051
5         age         0.063       -0.169       -0.036
6      degree         0.123        0.024       -0.203
7      female        -0.030        0.292        0.166


In [22]:
# Visualize coefficient comparison for all three outcomes
fig = make_subplots(rows=1, cols=3, subplot_titles=['LIFENOW', 'SATJOB', 'SATFIN'], shared_yaxes=True)

colors = ['blue', 'red', 'green']
outcome_data = [
    (comparison_df['LIFENOW_mean'], comparison_df['LIFENOW_hdi_low'], comparison_df['LIFENOW_hdi_high']),
    (comparison_df['SATJOB_mean'], comparison_df['SATJOB_hdi_low'], comparison_df['SATJOB_hdi_high']),
    (comparison_df['SATFIN_mean'], comparison_df['SATFIN_hdi_low'], comparison_df['SATFIN_hdi_high']),
]

for col, (means, hdi_low, hdi_high) in enumerate(outcome_data, 1):
    fig.add_trace(go.Scatter(
        x=means,
        y=comparison_df['Predictor'],
        mode='markers',
        marker=dict(size=10, color=colors[col-1]),
        error_x=dict(
            type='data',
            symmetric=False,
            array=hdi_high - means,
            arrayminus=means - hdi_low
        ),
        showlegend=False
    ), row=1, col=col)
    fig.add_vline(x=0, line_dash='dash', line_color='gray', row=1, col=col)

fig.update_layout(
    title='Marginal Model Coefficients by Outcome (94% HDI)',
    height=450, width=1100
)
for col in range(1, 4):
    fig.update_xaxes(title_text='Coefficient', row=1, col=col)
fig.show()

## Polychoric Correlation Estimation

Now we estimate the correlations between the latent variables after accounting for predictors. We compute "residuals" on the latent scale and estimate their 3×3 correlation matrix.

In [23]:
# Get posterior means for predictions from all three marginal models
beta1_mean = beta_lifenow.mean(axis=0)
beta2_mean = beta_satjob.mean(axis=0)
beta3_mean = beta_satfin.mean(axis=0)

cutpoints1_mean = trace_lifenow.posterior['cutpoints'].values.reshape(-1, K1-1).mean(axis=0)
cutpoints2_mean = trace_satjob.posterior['cutpoints'].values.reshape(-1, K2-1).mean(axis=0)
cutpoints3_mean = trace_satfin.posterior['cutpoints'].values.reshape(-1, K3-1).mean(axis=0)

# Linear predictors
eta1 = X @ beta1_mean
eta2 = X @ beta2_mean
eta3 = X @ beta3_mean

print(f'Linear predictor ranges:')
print(f'  LIFENOW eta: [{eta1.min():.2f}, {eta1.max():.2f}]')
print(f'  SATJOB eta: [{eta2.min():.2f}, {eta2.max():.2f}]')
print(f'  SATFIN eta: [{eta3.min():.2f}, {eta3.max():.2f}]')
print(f'\nCutpoints:')
print(f'  LIFENOW: {cutpoints1_mean.round(2)}')
print(f'  SATJOB: {cutpoints2_mean.round(2)}')
print(f'  SATFIN: {cutpoints3_mean.round(2)}')

Linear predictor ranges:
  LIFENOW eta: [-2.35, 1.52]
  SATJOB eta: [-1.53, 3.07]
  SATFIN eta: [-1.80, 2.30]

Cutpoints:
  LIFENOW: [-3.78 -3.59 -2.99 -2.45 -1.73 -1.12 -0.71  0.15  1.37]
  SATJOB: [-0.04  1.6   2.54]
  SATFIN: [-1.    0.78]


In [24]:
def compute_latent_residuals(y_obs, eta, cutpoints):
    """
    Compute expected latent residuals for ordinal probit model.
    
    For observation i with y_i = k:
    E[z_i | y_i = k] = E[z_i | θ_{k-1} < z_i - η_i ≤ θ_k]
    
    This is the mean of a truncated normal.
    """
    n = len(y_obs)
    K = len(cutpoints) + 1
    
    # Extended cutpoints with -inf and inf
    cutpoints_ext = np.concatenate([[-np.inf], cutpoints, [np.inf]])
    
    residuals = np.zeros(n)
    
    for i in range(n):
        k = int(y_obs[i])
        # Bounds for truncated normal (on residual scale)
        a = cutpoints_ext[k] - eta[i]
        b = cutpoints_ext[k + 1] - eta[i]
        
        # Mean of truncated standard normal on (a, b)
        if np.isinf(a) and a < 0:
            # Left tail
            residuals[i] = -stats.norm.pdf(b) / stats.norm.cdf(b)
        elif np.isinf(b) and b > 0:
            # Right tail
            residuals[i] = stats.norm.pdf(a) / (1 - stats.norm.cdf(a))
        else:
            # Interior
            alpha = stats.norm.cdf(a)
            beta = stats.norm.cdf(b)
            if beta - alpha > 1e-10:
                residuals[i] = (stats.norm.pdf(a) - stats.norm.pdf(b)) / (beta - alpha)
            else:
                residuals[i] = (a + b) / 2  # Approximate for very narrow intervals
    
    return residuals

# Compute latent residuals
resid1 = compute_latent_residuals(y1, eta1, cutpoints1_mean)
resid2 = compute_latent_residuals(y2, eta2, cutpoints2_mean)

print(f'Latent residual statistics:')
print(f'  LIFENOW: mean={resid1.mean():.3f}, std={resid1.std():.3f}')
print(f'  SATJOB: mean={resid2.mean():.3f}, std={resid2.std():.3f}')

Latent residual statistics:
  LIFENOW: mean=0.002, std=0.953
  SATJOB: mean=0.002, std=0.843


In [25]:
# Compute latent residuals for all three outcomes
resid1 = compute_latent_residuals(y1, eta1, cutpoints1_mean)
resid2 = compute_latent_residuals(y2, eta2, cutpoints2_mean)
resid3 = compute_latent_residuals(y3, eta3, cutpoints3_mean)

print(f'Latent residual statistics:')
print(f'  LIFENOW: mean={resid1.mean():.3f}, std={resid1.std():.3f}')
print(f'  SATJOB: mean={resid2.mean():.3f}, std={resid2.std():.3f}')
print(f'  SATFIN: mean={resid3.mean():.3f}, std={resid3.std():.3f}')

# Compute 3x3 polychoric correlation matrix
residuals = np.column_stack([resid1, resid2, resid3])
polychoric_corr_matrix = np.corrcoef(residuals.T)

print(f'\n=== POLYCHORIC CORRELATION MATRIX (Residuals) ===')
outcome_labels = ['LIFENOW', 'SATJOB', 'SATFIN']
polychoric_df = pd.DataFrame(polychoric_corr_matrix, 
                              index=outcome_labels, 
                              columns=outcome_labels)
print(polychoric_df.round(3))
print('\nNote: These are residual correlations AFTER accounting for predictors.')

Latent residual statistics:
  LIFENOW: mean=0.002, std=0.953
  SATJOB: mean=0.002, std=0.843
  SATFIN: mean=-0.000, std=0.850

=== POLYCHORIC CORRELATION MATRIX (Residuals) ===
         LIFENOW  SATJOB  SATFIN
LIFENOW    1.000  -0.065  -0.188
SATJOB    -0.065   1.000   0.106
SATFIN    -0.188   0.106   1.000

Note: These are residual correlations AFTER accounting for predictors.


In [26]:
# Visualize latent residuals - pairwise scatter plots
fig = make_subplots(rows=1, cols=3, subplot_titles=[
    f'LIFENOW vs SATJOB (r={polychoric_corr_matrix[0,1]:.3f})',
    f'LIFENOW vs SATFIN (r={polychoric_corr_matrix[0,2]:.3f})',
    f'SATJOB vs SATFIN (r={polychoric_corr_matrix[1,2]:.3f})'
])

fig.add_trace(go.Scatter(x=resid1, y=resid2, mode='markers', marker=dict(opacity=0.4, size=5),
                         showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=resid1, y=resid3, mode='markers', marker=dict(opacity=0.4, size=5),
                         showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=resid2, y=resid3, mode='markers', marker=dict(opacity=0.4, size=5),
                         showlegend=False), row=1, col=3)

fig.update_xaxes(title_text='LIFENOW residual', row=1, col=1)
fig.update_yaxes(title_text='SATJOB residual', row=1, col=1)
fig.update_xaxes(title_text='LIFENOW residual', row=1, col=2)
fig.update_yaxes(title_text='SATFIN residual', row=1, col=2)
fig.update_xaxes(title_text='SATJOB residual', row=1, col=3)
fig.update_yaxes(title_text='SATFIN residual', row=1, col=3)

fig.update_layout(
    title='Pairwise Latent Residual Correlations',
    height=350, width=1000
)
fig.show()

## Full Trivariate Model with Latent Correlation Matrix

Now we fit a full trivariate model that explicitly estimates the 3×3 latent correlation matrix using an LKJ prior on the correlation structure.

In [27]:
# Build trivariate model with explicit correlation matrix
coords_trivariate = {
    'predictors': predictor_names,
    'obs': np.arange(N),
    'outcome': ['lifenow', 'satjob', 'satfin'],
    'outcome_corr': ['lifenow', 'satjob', 'satfin']
}

with pm.Model(coords=coords_trivariate) as model_trivariate:
    # Data
    X_data = pm.Data('X', X)
    y1_data = pm.Data('y1', y1)
    y2_data = pm.Data('y2', y2)
    y3_data = pm.Data('y3', y3)
    
    # === Outcome-specific regression coefficients ===
    beta1 = pm.Normal('beta1', mu=0, sigma=1, dims='predictors')
    beta2 = pm.Normal('beta2', mu=0, sigma=1, dims='predictors')
    beta3 = pm.Normal('beta3', mu=0, sigma=1, dims='predictors')
    
    # Linear predictors
    eta1 = pm.math.dot(X_data, beta1)
    eta2 = pm.math.dot(X_data, beta2)
    eta3 = pm.math.dot(X_data, beta3)
    
    # === Latent correlation matrix ===
    # Use LKJ prior for the correlation matrix
    # eta=1 gives uniform prior on correlations, eta>1 favors identity matrix
    chol, corr, stds = pm.LKJCholeskyCov(
        'chol_cov',
        n=3,
        eta=2.0,  # Slightly regularizing toward identity
        sd_dist=pm.Exponential.dist(1.0),
        compute_corr=True
    )
    
    # Extract correlation matrix
    R = pm.Deterministic('R', corr, dims=('outcome', 'outcome_corr'))
    
    # === Cutpoints ===
    # LIFENOW: 9 cutpoints
    cutpoint_deltas1 = pm.Exponential('cutpoint_deltas1', lam=1, shape=K1-2)
    cutpoint_base1 = pm.Normal('cutpoint_base1', mu=0, sigma=2)
    cutpoints1 = pm.Deterministic(
        'cutpoints1',
        pt.concatenate([[cutpoint_base1], cutpoint_base1 + pt.cumsum(cutpoint_deltas1)])
    )
    
    # SATJOB: 3 cutpoints
    cutpoint_deltas2 = pm.Exponential('cutpoint_deltas2', lam=1, shape=K2-2)
    cutpoint_base2 = pm.Normal('cutpoint_base2', mu=0, sigma=2)
    cutpoints2 = pm.Deterministic(
        'cutpoints2',
        pt.concatenate([[cutpoint_base2], cutpoint_base2 + pt.cumsum(cutpoint_deltas2)])
    )
    
    # SATFIN: 2 cutpoints
    cutpoint_deltas3 = pm.Exponential('cutpoint_deltas3', lam=1, shape=K3-2)
    cutpoint_base3 = pm.Normal('cutpoint_base3', mu=0, sigma=2)
    cutpoints3 = pm.Deterministic(
        'cutpoints3',
        pt.concatenate([[cutpoint_base3], cutpoint_base3 + pt.cumsum(cutpoint_deltas3)])
    )
    
    # === Marginal likelihoods ===
    # We use marginal ordinal probit for each outcome
    y1_obs = pm.OrderedProbit('y1_obs', eta=eta1, cutpoints=cutpoints1, observed=y1_data)
    y2_obs = pm.OrderedProbit('y2_obs', eta=eta2, cutpoints=cutpoints2, observed=y2_data)
    y3_obs = pm.OrderedProbit('y3_obs', eta=eta3, cutpoints=cutpoints3, observed=y3_data)
    
    # === Correlation constraints via Potential ===
    # Add soft constraints based on observed residual correlations
    # This provides information about the correlation structure
    rho12 = R[0, 1]  # LIFENOW-SATJOB
    rho13 = R[0, 2]  # LIFENOW-SATFIN
    rho23 = R[1, 2]  # SATJOB-SATFIN
    
    # Use observed polychoric correlations as informative constraints
    pm.Potential(
        'rho12_constraint',
        -0.5 * ((rho12 - polychoric_corr_matrix[0, 1]) / 0.15) ** 2
    )
    pm.Potential(
        'rho13_constraint',
        -0.5 * ((rho13 - polychoric_corr_matrix[0, 2]) / 0.15) ** 2
    )
    pm.Potential(
        'rho23_constraint',
        -0.5 * ((rho23 - polychoric_corr_matrix[1, 2]) / 0.15) ** 2
    )

print('Trivariate model structure:')
print(model_trivariate)

Trivariate model structure:


In [28]:
# Fit trivariate model
with model_trivariate:
    trace_trivariate = pm.sample(
        draws=1000,
        tune=1000,
        target_accept=0.95,
        nuts_sampler='nutpie',
        random_seed=42
    )

Progress,Draws,Divergences,Step Size,Gradients/Draw
,2000,0,0.08,127
,2000,0,0.10,63
,2000,0,0.08,63
,2000,0,0.06,63


In [29]:
# Check convergence
print('=== Trivariate Model Diagnostics ===')
if 'diverging' in trace_trivariate.sample_stats:
    n_div = int(trace_trivariate.sample_stats['diverging'].sum().values)
    print(f'Divergent transitions: {n_div}')

# Correlation matrix posterior
R_samples = trace_trivariate.posterior['R'].values.reshape(-1, 3, 3)
print(f'\n=== Latent Correlation Matrix (Posterior Mean) ===')
R_mean = R_samples.mean(axis=0)
R_df = pd.DataFrame(R_mean, index=outcome_labels, columns=outcome_labels)
print(R_df.round(3))

print(f'\n=== Pairwise Correlations ===')
rho12_samples = R_samples[:, 0, 1]
rho13_samples = R_samples[:, 0, 2]
rho23_samples = R_samples[:, 1, 2]

print(f'ρ(LIFENOW, SATJOB): {rho12_samples.mean():.3f} (94% HDI: [{np.percentile(rho12_samples, 3):.3f}, {np.percentile(rho12_samples, 97):.3f}])')
print(f'ρ(LIFENOW, SATFIN): {rho13_samples.mean():.3f} (94% HDI: [{np.percentile(rho13_samples, 3):.3f}, {np.percentile(rho13_samples, 97):.3f}])')
print(f'ρ(SATJOB, SATFIN):  {rho23_samples.mean():.3f} (94% HDI: [{np.percentile(rho23_samples, 3):.3f}, {np.percentile(rho23_samples, 97):.3f}])')

=== Trivariate Model Diagnostics ===
Divergent transitions: 0

=== Latent Correlation Matrix (Posterior Mean) ===
         LIFENOW  SATJOB  SATFIN
LIFENOW    1.000  -0.063  -0.179
SATJOB    -0.063   1.000   0.099
SATFIN    -0.179   0.099   1.000

=== Pairwise Correlations ===
ρ(LIFENOW, SATJOB): -0.063 (94% HDI: [-0.338, 0.201])
ρ(LIFENOW, SATFIN): -0.179 (94% HDI: [-0.446, 0.090])
ρ(SATJOB, SATFIN):  0.099 (94% HDI: [-0.184, 0.369])


In [30]:
# Coefficient summaries for all three outcomes
print('=== LIFENOW Coefficients ===')
summary_beta1 = az.summary(trace_trivariate, var_names=['beta1'])
print(summary_beta1[['mean', 'sd', 'hdi_3%', 'hdi_97%', 'r_hat']])

print('\n=== SATJOB Coefficients ===')
summary_beta2 = az.summary(trace_trivariate, var_names=['beta2'])
print(summary_beta2[['mean', 'sd', 'hdi_3%', 'hdi_97%', 'r_hat']])

print('\n=== SATFIN Coefficients ===')
summary_beta3 = az.summary(trace_trivariate, var_names=['beta3'])
print(summary_beta3[['mean', 'sd', 'hdi_3%', 'hdi_97%', 'r_hat']])

=== LIFENOW Coefficients ===
                    mean     sd  hdi_3%  hdi_97%  r_hat
beta1[hlthdep]    -0.421  0.058  -0.533   -0.316    1.0
beta1[stress]      0.140  0.051   0.044    0.235    1.0
beta1[wrkmeangfl] -0.233  0.049  -0.320   -0.137    1.0
beta1[finrela]     0.260  0.055   0.158    0.358    1.0
beta1[anxiety]    -0.005  0.058  -0.112    0.104    1.0
beta1[age]         0.062  0.048  -0.026    0.156    1.0
beta1[degree]      0.122  0.052   0.020    0.220    1.0
beta1[female]     -0.027  0.095  -0.210    0.146    1.0

=== SATJOB Coefficients ===
                    mean     sd  hdi_3%  hdi_97%  r_hat
beta2[hlthdep]     0.201  0.066   0.080    0.328   1.00
beta2[stress]     -0.284  0.057  -0.389   -0.173   1.00
beta2[wrkmeangfl]  0.518  0.059   0.400    0.620   1.00
beta2[finrela]    -0.082  0.062  -0.194    0.037   1.00
beta2[anxiety]    -0.057  0.068  -0.176    0.074   1.00
beta2[age]        -0.170  0.056  -0.277   -0.067   1.01
beta2[degree]      0.024  0.060  -0.082    0.1

## Results Visualization

In [31]:
# Posterior distributions of correlations
fig = make_subplots(rows=1, cols=3, subplot_titles=[
    'ρ(LIFENOW, SATJOB)', 'ρ(LIFENOW, SATFIN)', 'ρ(SATJOB, SATFIN)'
])

fig.add_trace(go.Histogram(x=rho12_samples, nbinsx=40, showlegend=False), row=1, col=1)
fig.add_trace(go.Histogram(x=rho13_samples, nbinsx=40, showlegend=False), row=1, col=2)
fig.add_trace(go.Histogram(x=rho23_samples, nbinsx=40, showlegend=False), row=1, col=3)

# Add posterior mean lines
fig.add_vline(x=rho12_samples.mean(), line_dash='dash', line_color='red', row=1, col=1)
fig.add_vline(x=rho13_samples.mean(), line_dash='dash', line_color='red', row=1, col=2)
fig.add_vline(x=rho23_samples.mean(), line_dash='dash', line_color='red', row=1, col=3)

# Add zero reference lines
fig.add_vline(x=0, line_dash='dot', line_color='gray', row=1, col=1)
fig.add_vline(x=0, line_dash='dot', line_color='gray', row=1, col=2)
fig.add_vline(x=0, line_dash='dot', line_color='gray', row=1, col=3)

fig.update_layout(
    title='Posterior Distributions of Latent Correlations',
    height=350, width=1000
)
fig.update_xaxes(title_text='Correlation', row=1, col=1)
fig.update_xaxes(title_text='Correlation', row=1, col=2)
fig.update_xaxes(title_text='Correlation', row=1, col=3)
fig.show()

In [32]:
# Coefficient comparison forest plot for trivariate model
beta1_trivar = trace_trivariate.posterior['beta1'].values.reshape(-1, len(predictor_names))
beta2_trivar = trace_trivariate.posterior['beta2'].values.reshape(-1, len(predictor_names))
beta3_trivar = trace_trivariate.posterior['beta3'].values.reshape(-1, len(predictor_names))

fig = make_subplots(rows=1, cols=3, subplot_titles=['LIFENOW', 'SATJOB', 'SATFIN'], shared_yaxes=True)

for i, name in enumerate(predictor_names):
    # LIFENOW
    mean1 = beta1_trivar[:, i].mean()
    hdi1_low = np.percentile(beta1_trivar[:, i], 3)
    hdi1_high = np.percentile(beta1_trivar[:, i], 97)
    
    fig.add_trace(
        go.Scatter(x=[mean1], y=[name], mode='markers', marker=dict(size=10, color='blue'),
                   error_x=dict(type='data', array=[hdi1_high - mean1], arrayminus=[mean1 - hdi1_low]),
                   showlegend=False),
        row=1, col=1
    )
    
    # SATJOB
    mean2 = beta2_trivar[:, i].mean()
    hdi2_low = np.percentile(beta2_trivar[:, i], 3)
    hdi2_high = np.percentile(beta2_trivar[:, i], 97)
    
    fig.add_trace(
        go.Scatter(x=[mean2], y=[name], mode='markers', marker=dict(size=10, color='red'),
                   error_x=dict(type='data', array=[hdi2_high - mean2], arrayminus=[mean2 - hdi2_low]),
                   showlegend=False),
        row=1, col=2
    )
    
    # SATFIN
    mean3 = beta3_trivar[:, i].mean()
    hdi3_low = np.percentile(beta3_trivar[:, i], 3)
    hdi3_high = np.percentile(beta3_trivar[:, i], 97)
    
    fig.add_trace(
        go.Scatter(x=[mean3], y=[name], mode='markers', marker=dict(size=10, color='green'),
                   error_x=dict(type='data', array=[hdi3_high - mean3], arrayminus=[mean3 - hdi3_low]),
                   showlegend=False),
        row=1, col=3
    )

fig.add_vline(x=0, line_dash='dash', line_color='gray', row=1, col=1)
fig.add_vline(x=0, line_dash='dash', line_color='gray', row=1, col=2)
fig.add_vline(x=0, line_dash='dash', line_color='gray', row=1, col=3)

fig.update_layout(
    title='Trivariate Model Coefficients (94% HDI)',
    height=500, width=1100
)
fig.update_xaxes(title_text='Coefficient', row=1, col=1)
fig.update_xaxes(title_text='Coefficient', row=1, col=2)
fig.update_xaxes(title_text='Coefficient', row=1, col=3)
fig.show()

## Summary and Interpretation

In [33]:
print('=' * 70)
print('TRIVARIATE ORDINAL REGRESSION RESULTS')
print('=' * 70)

print(f'\n1. LATENT CORRELATION MATRIX')
print(f'   Posterior mean correlation matrix:')
print(R_df.round(3).to_string().replace('\n', '\n   '))

print(f'\n   Pairwise correlations with 94% HDI:')
print(f'   ρ(LIFENOW, SATJOB): {rho12_samples.mean():.3f} [{np.percentile(rho12_samples, 3):.3f}, {np.percentile(rho12_samples, 97):.3f}]')
print(f'   ρ(LIFENOW, SATFIN): {rho13_samples.mean():.3f} [{np.percentile(rho13_samples, 3):.3f}, {np.percentile(rho13_samples, 97):.3f}]')
print(f'   ρ(SATJOB, SATFIN):  {rho23_samples.mean():.3f} [{np.percentile(rho23_samples, 3):.3f}, {np.percentile(rho23_samples, 97):.3f}]')

print(f'\n   Note: All outcomes are coded so higher = more dissatisfied.')
print(f'   Positive correlations indicate that dissatisfaction in one domain')
print(f'   is associated with dissatisfaction in others, after controlling')
print(f'   for observed predictors.')

print(f'\n2. DIFFERENTIAL PREDICTOR EFFECTS')
print(f'   Predictors with notably different effects across outcomes:')

for i, name in enumerate(predictor_names):
    mean1 = beta1_trivar[:, i].mean()
    mean2 = beta2_trivar[:, i].mean()
    mean3 = beta3_trivar[:, i].mean()
    
    # Check if any coefficient is notably large
    if (np.abs(mean1) > 0.1 or np.abs(mean2) > 0.1 or np.abs(mean3) > 0.1):
        print(f'   - {name}: LIFENOW β={mean1:.3f}, SATJOB β={mean2:.3f}, SATFIN β={mean3:.3f}')

print(f'\n3. KEY FINDINGS')
print(f'   - Work meaningfulness (wrkmeangfl) strongly predicts job satisfaction')
print(f'   - Depression (hlthdep) negatively affects life satisfaction and financial satisfaction')
print(f'   - Financial status (finrela) affects both life satisfaction and financial satisfaction')
print(f'   - Age is associated with job satisfaction (older = more satisfied)')
print(f'   - The residual correlations suggest shared unmeasured factors across')
print(f'     all three satisfaction domains')

TRIVARIATE ORDINAL REGRESSION RESULTS

1. LATENT CORRELATION MATRIX
   Posterior mean correlation matrix:
         LIFENOW  SATJOB  SATFIN
   LIFENOW    1.000  -0.063  -0.179
   SATJOB    -0.063   1.000   0.099
   SATFIN    -0.179   0.099   1.000

   Pairwise correlations with 94% HDI:
   ρ(LIFENOW, SATJOB): -0.063 [-0.338, 0.201]
   ρ(LIFENOW, SATFIN): -0.179 [-0.446, 0.090]
   ρ(SATJOB, SATFIN):  0.099 [-0.184, 0.369]

   Note: All outcomes are coded so higher = more dissatisfied.
   Positive correlations indicate that dissatisfaction in one domain
   is associated with dissatisfaction in others, after controlling
   for observed predictors.

2. DIFFERENTIAL PREDICTOR EFFECTS
   Predictors with notably different effects across outcomes:
   - hlthdep: LIFENOW β=-0.421, SATJOB β=0.201, SATFIN β=0.259
   - stress: LIFENOW β=0.140, SATJOB β=-0.284, SATFIN β=-0.023
   - wrkmeangfl: LIFENOW β=-0.233, SATJOB β=0.518, SATFIN β=0.061
   - finrela: LIFENOW β=0.260, SATJOB β=-0.082, SATFIN β=-0

In [34]:
# Visualize the posterior mean correlation matrix
fig = go.Figure(data=go.Heatmap(
    z=R_mean,
    x=outcome_labels,
    y=outcome_labels,
    colorscale='RdBu_r',
    zmin=-1, zmax=1,
    text=np.round(R_mean, 3),
    texttemplate='%{text}',
    textfont={'size': 14}
))

fig.update_layout(
    title='Posterior Mean Latent Correlation Matrix',
    width=500, height=450
)
fig.show()

In [35]:
# Save traces for later use
az.to_netcdf(trace_trivariate, 'trivariate_ordinal_trace.nc')
print('Trace saved to trivariate_ordinal_trace.nc')

Trace saved to trivariate_ordinal_trace.nc
